# AMML Ranking Competition 2026 — solution

**System:** Kaggle Notebook GPU (preferred) or local Windows/Linux CPU  
**Metric:** NDCG@10  
**Pipeline:** lexical/structural features → CatBoost YetiRank → fine-tune CrossEncoder on competition pairs → within-query blend → submission

Uses **only** competition train labels for fine-tuning (no external labeled IR datasets).

In [ ]:
import os, sys, subprocess, platform
from pathlib import Path

ON_KAGGLE = Path('/kaggle/input').exists()
print('platform:', platform.platform())
print('on_kaggle:', ON_KAGGLE)

if ON_KAGGLE:
    ROOT = Path('/kaggle/working/amml-ranking-competition-2026')
    DATA = Path('/kaggle/input/amml-ranking-competition-2026')
    # Pull latest public code (internet must be ON)
    if not ROOT.exists():
        subprocess.check_call(['git', 'clone', '--depth', '1',
            'https://github.com/VenistorAlishi/amml-ranking-competition-2026.git', str(ROOT)])
    else:
        subprocess.call(['git', '-C', str(ROOT), 'pull', '--ff-only'])
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
        'rank-bm25==0.2.2', 'catboost==1.2.7'])
else:
    ROOT = Path('.').resolve()
    DATA = ROOT / 'data'

sys.path.insert(0, str(ROOT))
print('ROOT', ROOT)
print('DATA', DATA)
assert (DATA / 'train.csv').exists(), DATA

In [ ]:
import numpy as np
import pandas as pd
import torch
from pathlib import Path

from src.preprocessing import prepare_frame
from src.features import assemble_features
from src.train import cross_validate, train_full, predict, get_feature_cols
from src.ce_train import train_cross_encoder, predict_cross_encoder
from src.metrics import mean_ndcg_at_k
from src.pipeline import blend_frames

print('cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')

OUT = Path('/kaggle/working') if ON_KAGGLE else (ROOT / 'results')
MODELS = Path('/kaggle/working/models') if ON_KAGGLE else (ROOT / 'models')
OUT.mkdir(parents=True, exist_ok=True)
MODELS.mkdir(parents=True, exist_ok=True)

train = prepare_frame(pd.read_csv(DATA / 'train.csv'))
test = prepare_frame(pd.read_csv(DATA / 'test.csv'))
print(train.shape, test.shape, 'pos_rate', train.relevance.mean())

In [ ]:
print('Building features...')
feat_tr, feat_te = assemble_features(train, test)
labels = train['relevance']
print('features', feat_tr.shape)

In [ ]:
print('CatBoost YetiRank GroupKFold CV...')
cv_cb, oof_cb = cross_validate(feat_tr, labels, n_splits=5)
model = train_full(feat_tr, labels, MODELS, best_iterations=1500)
feature_cols = get_feature_cols(feat_tr.assign(relevance=labels))
cb_te = predict(model, feat_te, feature_cols)
print('CatBoost CV NDCG@10', cv_cb)

In [ ]:
# Fine-tune CrossEncoder on competition data only (GPU on Kaggle)
ce_epochs = 3 if torch.cuda.is_available() else 1
ce_bs = 64 if torch.cuda.is_available() else 16
ce_model = 'cointegrated/rubert-tiny2'

print(f'Fine-tuning CE {ce_model} epochs={ce_epochs} bs={ce_bs}')
arts = train_cross_encoder(
    train, MODELS,
    model_name=ce_model,
    epochs=ce_epochs,
    batch_size=ce_bs,
    max_len=192,
    lr=2e-5,
)
ce_tr = predict_cross_encoder(arts, train, batch_size=128 if torch.cuda.is_available() else 32)
ce_te = predict_cross_encoder(arts, test, batch_size=128 if torch.cuda.is_available() else 32)
print('CE train score stats', float(ce_tr.mean()), float(ce_tr.std()))

In [ ]:
# Honest blend: search CE weight carefully (CE preds on train are in-sample → keep weight small)
best_w, best_s = 0.0, -1.0
for w in [0.0, 0.1, 0.15, 0.2, 0.25, 0.35]:
    pred = blend_frames(train['query_id'].to_numpy(), [oof_cb, ce_tr], [1 - w, w])
    scored = train[['id', 'query_id', 'relevance']].copy()
    scored['prediction'] = pred
    # When w>0 this overestimates because ce_tr is not OOF; still useful for ordering small weights
    s = mean_ndcg_at_k(scored)
    # Prefer CB OOF when CE boost looks unrealistically large
    adj = s if w == 0 else min(s, cv_cb + 0.02)
    print(f'w_ce={w:.2f} raw={s:.6f} adj={adj:.6f}')
    if adj > best_s:
        best_s, best_w = adj, w

print('selected w_ce', best_w, 'adj', best_s)
te_pred = blend_frames(test['query_id'].to_numpy(), [cb_te, ce_te], [1 - best_w, best_w])

sub = test[['id']].copy()
sub['prediction'] = te_pred
sub_path = OUT / 'submission.csv'
sub.to_csv(sub_path, index=False)
print('wrote', sub_path, len(sub))
sub.head()

In [ ]:
# Also dump CatBoost-only for comparison
sub_cb = test[['id']].copy()
sub_cb['prediction'] = cb_te
sub_cb.to_csv(OUT / 'submission_cb_only.csv', index=False)
print('cb-only saved')

# On Kaggle, attach competition and click Submit from the notebook output,
# or use the lines below if kaggle CLI credentials are available in secrets.
if ON_KAGGLE:
    print('Submit via Kaggle UI from /kaggle/working/submission.csv')
else:
    print('Local run done — submit with: kaggle competitions submit -c amml-ranking-competition-2026 -f results/submission.csv -m "ce+catboost"')